# صوت AI جاهز بالعربية والإنجليزية
**اكتب النص → اختر اللغة والصوت → شغّل → اسمع ونزّل WAV.**
لا تحتاج تسجيل صوتك أو تدريب نموذج. اختر T4 GPU من إعدادات Runtime لتسريع التشغيل؛ CPU متاح لكنه أبطأ.

حمّل ZIP للمستودع من GitHub وارفعه في الخلية التالية. هذا رفع للكود فقط.
الأصوات جاهزة من XTTS-v2. ترخيص أوزانه ومخرجاته CPML للاستخدام غير التجاري: [المصدر](https://huggingface.co/coqui/XTTS-v2/blob/main/LICENSE.txt).
الوضوح والنطق يتغيران بحسب النص والصوت المختار؛ استمع للنتيجة قبل استخدامها.

In [ ]:
from pathlib import Path
import zipfile
import os
import subprocess
import sys
from google.colab import files, drive

# Download Code -> Download ZIP from your private GitHub repository, then upload it here.
if not Path('/content/Voice2Reel/pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    destination = Path('/content/repo_upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    import shutil
    shutil.copytree(projects[0].parent, '/content/Voice2Reel')
    del uploaded
os.chdir('/content/Voice2Reel')
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Voice2Reel')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', WORK)


## تثبيت المكتبات — مرة واحدة في الجلسة

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.8'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-env'], check=True)
PY = '/content/v2r-env/bin/python'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)
def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)
pip_install('torch==2.8.0', 'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
pip_install('-r', 'requirements/voice.txt')
run('-c', "import torch; print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower)')")
# Isolated Python environment: no kernel restart or changes to Colab's preinstalled packages.

freeze = subprocess.check_output([sys.executable, "-m", "uv", "pip", "freeze", "--python", PY], text=True)
(WORK / "00_ready_voice_ar_en_environment.txt").write_text(freeze)


## اختيار اللغة والصوت والنص

In [ ]:
#@title إعدادات التعليق الصوتي
LANGUAGE = "ar" #@param ["ar", "en"]
SPEAKER = "Claribel Dervla" #@param ["Claribel Dervla", "Andrew Chipper", "Ana Florence", "Daisy Studious"]
TEXT = "مرحبًا بكم! في هذا الفيديو سنتعلم كيف نحول فكرة بسيطة إلى خطوات واضحة." #@param {type:"string"}
# For English: choose en and enter e.g. "Welcome! Let's turn a simple idea into clear steps."
text_file = WORK / 'ready_voice_text.txt'
text_file.write_text(TEXT, encoding='utf-8')
OUTPUT = WORK / f'ready_voice_{LANGUAGE}.wav'
run('-m', 'voice2reel.voice.synthesize', '--text', text_file, '--language', LANGUAGE,
    '--speaker', SPEAKER, '--output', OUTPUT)


## الاستماع وتنزيل الملف

In [ ]:
from IPython.display import Audio, display
display(Audio(str(OUTPUT)))
files.download(str(OUTPUT))
print('Audio and metadata saved in:', WORK)


## اختياري: قراءة سكربت جاهز
إذا كان لديك script.json من دفتر السكربت، شغّل الخلية التالية. تُقرأ اللغة من script.language.
الصوت المختار أعلاه يُستخدم تلقائيا، وتُحفظ توقيتات المقاطع بجانب WAV.

In [ ]:
SCRIPT = WORK / 'script.json'
if SCRIPT.is_file():
    run('-m', 'voice2reel.voice.synthesize', '--script', SCRIPT, '--speaker', SPEAKER,
        '--output', WORK / 'script_narration.wav')
    display(Audio(str(WORK / 'script_narration.wav')))
else:
    print('No script.json yet. You can already generate speech using the text cell above.')


لتغيير النص أو اللغة، عدّل خلية الإعدادات وأعد تشغيلها ثم خلية الاستماع.
للاطلاع على جميع الأصوات: `python -m voice2reel.voice.synthesize --list-speakers`.
المخرجات صوتية؛ إنشاء الفيديو ما زال مرحلة مستقلة في المشروع.